# Industrial Leak Incident Detection: Training & Evaluation Pipeline

This notebook implements an end-to-end machine learning pipeline for detecting and classifying industrial leak incidents:
- **Target Column**: `incident_label`
  - `0`: Normal (~70%)
  - `1`: Warning (~15%)
  - `2`: Leak Suspected (~10%)
  - `3`: Confirmed Leak (~5%)

### Workflow Steps:
1. **Load Data**: Load raw industrial sensor dataset (`industrial_leak_training_50k.csv`).
2. **Train/Test Split**: Perform a stratified 80/20 split to maintain class ratios.
3. **Feature Preparation**: Drop target-leaking and identifier columns, label-encode categorical features, and impute missing numeric values with training medians.
4. **Model Training**: Train a `RandomForestClassifier` with balanced class weights on the training set only.
5. **Evaluation**: Predict on unseen test set, report accuracy, full classification metrics, confusion matrix, top 10 feature importances, and plot a confusion matrix heatmap.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set plot styling
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

In [ ]:
# Load raw dataset
import os
dataset_path = "dataset/industrial_leak_training_50k.csv" if os.path.exists("dataset/industrial_leak_training_50k.csv") else "industrial_leak_training_50k.csv"
df = pd.read_csv(dataset_path)

print(f"Loaded '{dataset_path}': {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## Train/Test Split

We perform a stratified 80% train / 20% test split based on `incident_label` with `random_state=42` to preserve class proportions across splits.

In [ ]:
target_col = "incident_label"
label_names = {
    0: "Normal (0)",
    1: "Warning (1)",
    2: "Leak Suspected (2)",
    3: "Confirmed Leak (3)"
}

# Stratified 80/20 train/test split
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df[target_col]
)

# Print class distributions
def print_distribution(data, name):
    counts = data[target_col].value_counts().sort_index()
    pcts = (data[target_col].value_counts(normalize=True).sort_index() * 100).round(2)
    summary_df = pd.DataFrame({
        "Class Name": [label_names[i] for i in counts.index],
        "Count": counts.values,
        "Percentage (%)": pcts.values
    }, index=counts.index)
    print(f"=== {name} ({len(data):,} rows) ===")
    print(summary_df.to_string(index=False))
    print()

print_distribution(train_df, "Train Set")
print_distribution(test_df, "Test Set")

## Feature Preparation

In this step:
1. **Drop Leaky / Identifier Columns**:
   - `risk_class`, `risk_score`, `leak_location`, `leak_severity`, `confirmed_by` (leak the target).
   - `timestamp`, `plant_id`, `plant_name`, `process_unit_id`, `process_unit_name`, `equipment_id`, `equipment_name` (pure identifiers / text).
2. **Numeric Missing Value Imputation**:
   - Compute column medians on training data **only** to prevent data snooping.
   - Apply the calculated medians to both train and test splits.
3. **Categorical Encoding**:
   - Label-encode `equipment_type`, `process_type`, `fuel_or_material_type`, `shift`, and `maintenance_status`.

In [ ]:
# Columns to drop before training
leak_and_id_cols = [
    "risk_class", "risk_score", "leak_location", "leak_severity",
    "confirmed_by", "timestamp", "plant_id", "plant_name",
    "process_unit_id", "process_unit_name", "equipment_id", "equipment_name"
]

# Separate features and target
cols_to_drop = leak_and_id_cols + [target_col]
X_train = train_df.drop(columns=cols_to_drop).copy()
y_train = train_df[target_col].copy()

X_test = test_df.drop(columns=cols_to_drop).copy()
y_test = test_df[target_col].copy()

categorical_cols = [
    "equipment_type", "process_type", "fuel_or_material_type",
    "shift", "maintenance_status"
]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

# 1. Impute missing numeric values using training set medians
train_medians = X_train[numeric_cols].median()
X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
X_test[numeric_cols] = X_test[numeric_cols].fillna(train_medians)

# 2. Label-encode categorical features
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

print(f"Feature matrix prepared:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"  Remaining features ({len(X_train.columns)}): {list(X_train.columns)}")

## Model Training

Train a `RandomForestClassifier` on the training set only with:
- `n_estimators = 200`
- `class_weight = 'balanced'` (to handle class imbalance: 70% / 15% / 10% / 5%)
- `random_state = 42`

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training RandomForestClassifier on train set ONLY...")
rf_model.fit(X_train, y_train)
print("Model training complete!")

## Evaluation

Evaluate the trained model on the unseen test split (`X_test`, `y_test`).

In [ ]:
# Predict on test set
y_pred = rf_model.predict(X_test)

# 1. Overall Accuracy
acc = accuracy_score(y_test, y_pred)
print("=" * 60)
print(f"OVERALL ACCURACY: {acc:.4f} ({acc * 100:.2f}%)")
print("=" * 60)

# 2. Classification Report
target_display_names = [label_names[i] for i in sorted(label_names.keys())]
print("\nCLASSIFICATION REPORT:")
print("-" * 60)
print(classification_report(y_test, y_pred, target_names=target_display_names, digits=4))

# 3. Confusion Matrix as a labeled DataFrame
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {name}" for name in target_display_names],
    columns=[f"Pred {name}" for name in target_display_names]
)
print("CONFUSION MATRIX (Rows = Actual, Columns = Predicted):")
print("-" * 60)
display(cm_df) if 'display' in globals() else print(cm_df)

# 4. Top 10 Feature Importances
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": importances,
    "Importance (%)": (importances * 100).round(2)
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

top10_df = feat_imp_df.head(10).copy()
top10_df.index = range(1, 11)
top10_df.index.name = "Rank"

print("\nTOP 10 MOST IMPORTANT FEATURES:")
print("=" * 60)
display(top10_df) if 'display' in globals() else print(top10_df)

### Confusion Matrix Visualization

Visualizing the confusion matrix both as raw counts and as row-normalized percentages (Recall per class).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Raw counts heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_display_names,
    yticklabels=target_display_names,
    ax=axes[0],
    cbar=False
)
axes[0].set_title("Confusion Matrix (Counts)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Predicted Class", fontsize=12)
axes[0].set_ylabel("Actual Class", fontsize=12)

# 2. Normalized percentages heatmap (Recall per class)
cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2%",
    cmap="Blues",
    xticklabels=target_display_names,
    yticklabels=target_display_names,
    ax=axes[1]
)
axes[1].set_title("Normalized Confusion Matrix (Recall %)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Predicted Class", fontsize=12)
axes[1].set_ylabel("Actual Class", fontsize=12)

plt.tight_layout()
plt.show()